# Practical 10: Weather API and Product Crawler

**Aim:** Fetch weather for a city and collect product names, prices, and links from a listing page.

This notebook uses the keyless Open-Meteo geocoding and forecast APIs, plus Books to Scrape, a bookstore sandbox for crawler practice. Install `requests` and `beautifulsoup4` if needed: `python -m pip install requests beautifulsoup4`. An internet connection is required. Run the cells from top to bottom.


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"
SHOP_URL = "https://books.toscrape.com/"


## 1. Fetch weather for a city

The first request finds the city's coordinates. The second requests current temperature, wind speed, and a weather code. Change `city` to try another place.


In [ ]:
def show_weather(city):
    place = requests.get(GEOCODE_URL, params={"name": city, "count": 1}, timeout=10).json()
    if not place.get("results"):
        print("City not found")
        return
    place = place["results"][0]
    params = {
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "current": "temperature_2m,wind_speed_10m,weather_code",
        "timezone": "auto",
    }
    current = requests.get(WEATHER_URL, params=params, timeout=10).json()["current"]
    descriptions = {0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
                    45: "Fog", 61: "Rain", 71: "Snow", 80: "Rain showers", 95: "Thunderstorm"}
    print("Weather in", place["name"], place.get("country", ""))
    print("Temperature:", current["temperature_2m"], "°C")
    print("Wind speed:", current["wind_speed_10m"], "km/h")
    print("Description:", descriptions.get(current["weather_code"], "Other conditions"))

city = "Mumbai"
show_weather(city)


## 2. Crawl product details

The selector finds product cards on the first page. Each product's relative link is converted to a full URL.


In [ ]:
page = requests.get(SHOP_URL, timeout=10)
page.raise_for_status()
soup = BeautifulSoup(page.text, "html.parser")

for product in soup.select("article.product_pod"):
    link = product.select_one("h3 a")
    name = link["title"]
    price = product.select_one(".price_color").get_text(strip=True)
    print(name, "|", price, "|", urljoin(SHOP_URL, link["href"]))


## Viva questions with answers

1. **What is an API?** A way for one program to request data or services from another program.
2. **What is JSON?** A text format commonly used to send structured data over an API.
3. **Why geocode a city first?** The forecast API needs latitude and longitude coordinates.
4. **What does `timeout=10` do?** It stops the request from waiting indefinitely if the server does not respond.
5. **What does a weather code represent?** A numeric label for a condition such as clear sky, rain, or snow.
6. **What is web crawling?** Automatically requesting web pages and collecting selected information from them.
7. **What does BeautifulSoup do?** It parses HTML so the program can find elements and text.
8. **What is a CSS selector?** A pattern for selecting HTML elements, such as `article.product_pod`.
9. **Why use `urljoin`?** It turns a product's relative link into a complete URL.
10. **What details does this crawler collect?** Product title, price, and link from the first listing page.
